In [52]:
import sys
import os
import torch
import numpy as np
import yaml

sys.path.append(os.path.join(os.getcwd(), '..'))

from m2a_transformer_inference import decode_output
from preprocess.preprocess_midi2pt_dataset import preprocess_midi, DURATION_TEMPLATES
from m2a_transformer_inference import decompress
from m2a_transformer import RoFormerSymbolicTransformer, EOS_TOKEN, PAD_TOKEN


In [53]:
import pytorch_lightning as L
import torch.nn as nn
import torch.nn.functional as F
from transformers.models.roformer.modeling_roformer import RoFormerModel, RoFormerConfig, RoFormerEncoder

N_NORMAL_TOKENS = 3202
N_TOKENS = N_NORMAL_TOKENS + 3
SOS_TOKEN = N_NORMAL_TOKENS
EOS_TOKEN = N_NORMAL_TOKENS + 1
PAD_TOKEN = N_NORMAL_TOKENS + 2

def fill_with_neg_inf(t):
    return t.float().fill_(float("-inf")).type_as(t)
class OldM2ATransformer(L.LightningModule):
    def __init__(self, hidden_size, num_layers, num_attention_heads, intermediate_size, 
                 local_model_num_layers, local_model_num_attention_heads, local_model_intermediate_size, **kwargs):
        super().__init__()
        self.save_hyperparameters()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.local_model_num_layers = local_model_num_layers
        self.local_model_num_attention_heads = local_model_num_attention_heads
        self.local_model_intermediate_size = local_model_intermediate_size

        main_roformer_config = RoFormerConfig(
            hidden_size=self.hidden_size,
            num_hidden_layers=self.num_layers,
            num_attention_heads=self.num_attention_heads,
            intermediate_size=self.intermediate_size,
            hidden_act="gelu",
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.model = self.get_base_model(main_roformer_config)
        local_encoder_config = local_decoder_config = RoFormerConfig(
            hidden_size=self.hidden_size,
            num_hidden_layers=self.local_model_num_layers,
            num_attention_heads=self.local_model_num_attention_heads,
            intermediate_size=self.local_model_intermediate_size,
            hidden_act="gelu",
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.local_embedding = nn.Embedding(N_TOKENS, self.hidden_size)
        self.token_type_embeddings = nn.Embedding(2, self.hidden_size)
        with torch.no_grad():
            self.token_type_embeddings.weight.mul_(2.0)
        self.local_encoder = RoFormerEncoder(local_encoder_config)
        self.local_decoder = RoFormerEncoder(local_decoder_config)
        self.final_decoder = nn.Linear(self.hidden_size, N_TOKENS)
        self.global_sos = nn.Parameter(torch.randn(self.hidden_size))
        self._future_mask = torch.empty(0)

    def get_base_model(self, config):
        return RoFormerEncoder(config)
    
    # All other methods from RoFormerSymbolicTransformer are copied here verbatim
    def local_encode(self, x, token_type_ids):
        batch_size, seq_len, subseq_len = x.shape
        x = x.view(-1, subseq_len)
        x = torch.cat([
            torch.full((x.shape[0], 1), SOS_TOKEN, dtype=torch.long, device=x.device),
            x
        ], dim=-1)
        mask = x != PAD_TOKEN
        word_emb = self.local_embedding(x)
        type_emb = self.token_type_embeddings(token_type_ids)
        type_emb = type_emb.view(batch_size*seq_len, word_emb.shape[1], -1)
        emb = word_emb + type_emb
        h = self.local_encoder(emb, encoder_attention_mask=mask)[0]
        return h[:, 0], emb[:, :-1]

    def local_decode(self, h, emb):
        batch_size, subseq_len, _ = emb.shape
        h = h.view(batch_size, 1, -1)
        emb = torch.cat([h, emb[:, 1:]], dim=1)
        h = self.local_decoder(emb, attention_mask=self.buffered_future_mask(emb))[0]
        return self.final_decoder(h)

    def buffered_future_mask(self, tensor):
        dim = tensor.size(1)
        if (
                self._future_mask.size(0) == 0
                or (not self._future_mask.device == tensor.device)
                or self._future_mask.size(0) < dim
        ):
            self._future_mask = torch.triu(
                fill_with_neg_inf(torch.zeros([dim, dim])), 1
            )
        self._future_mask = self._future_mask.to(tensor)
        return self._future_mask[:dim, :dim]

    def forward(self, x):
        batch_size, seq_len, subseq_len = x.shape
        assert seq_len % 2 == 0, "Expected even number of frames (2*S interleaved)."
        idx = torch.arange(seq_len, device=x.device)
        frame_type = (idx % 2 == 0).long()
        token_type_ids = frame_type.unsqueeze(0).unsqueeze(-1).expand(batch_size, seq_len, subseq_len)
        sos_type = frame_type.unsqueeze(0).unsqueeze(-1).expand(batch_size, seq_len, 1)
        token_type_ids = torch.cat([sos_type, token_type_ids], dim=-1)
        h, emb= self.local_encode(x, token_type_ids)
        h = h.view(batch_size, seq_len, -1)
        sos = self.global_sos.view(1, 1, -1).repeat(batch_size, 1, 1)
        h = torch.cat([sos, h[:, :-1]], dim=1)
        h = self.model(h, attention_mask=self.buffered_future_mask(h), interleave_pos=True)[0]
        return self.local_decode(h, emb)

    def preprocess(self, x, pitch_shift, y=None):
        batch_size, seq_length, subseq_length = x.shape
        x = x.long().view(batch_size, seq_length, subseq_length // 3, 3)
        x_processed = torch.zeros(batch_size, seq_length, subseq_length // 3, 2, dtype=torch.long, device=x.device)
        pad_indices = x[:, :, :, 1] == 255
        eos_indices = x[:, :, :, 0] == 254
        is_not_drum = x[:, :, :, 0] != 127
        x_processed[:, :, :, 0] = 0
        x_processed[:, :, :, 1] = x[:, :, :, 1] + (x[:, :, :, 2]) * 128 + 2 + pitch_shift[:, None, None] * is_not_drum
        x_processed[pad_indices] = PAD_TOKEN
        x_processed[:, :, :, 0][eos_indices] = EOS_TOKEN
        if y is None:
            return x_processed.view(batch_size, seq_length, subseq_length // 3 * 2)
        else:
            batch_size_y, seq_length_y, subseq_length_y = y.shape
            y = y.long().view(batch_size_y, seq_length_y, subseq_length_y // 3, 3)
            y_processed = torch.zeros(batch_size_y, seq_length_y, subseq_length_y // 3, 2, dtype=torch.long, device=y.device)
            pad_indices_y = y[:, :, :, 1] == 255
            eos_indices_y = y[:, :, :, 0] == 254
            is_not_drum_y = y[:, :, :, 0] != 127
            y_processed[:, :, :, 0] = 1
            y_processed[:, :, :, 1] = y[:, :, :, 1] + (y[:, :, :, 2]) * 128 + 2 + pitch_shift[:, None, None] * is_not_drum_y
            y_processed[pad_indices_y] = PAD_TOKEN
            y_processed[:, :, :, 0][eos_indices_y] = EOS_TOKEN
            return x_processed.view(batch_size, seq_length, subseq_length // 3 * 2), y_processed.view(batch_size_y, seq_length_y, subseq_length_y // 3 * 2)

    def loss(self, x_mel, x_acc, pitch_shift):
        x_mel, x_acc = self.preprocess(x_mel, pitch_shift, y=x_acc)
        batch_size, seq_len, subseq_len = x_mel.shape
        stacked = torch.stack([x_acc, x_mel], dim=2)
        x = stacked.view(batch_size, seq_len * 2, subseq_len)
        x_target = x.clone()
        idx = torch.arange(seq_len * 2, device=x.device)
        mel_mask = (idx % 2 == 1).unsqueeze(0).unsqueeze(-1)
        mel_mask = mel_mask.expand(batch_size, seq_len * 2, subseq_len)
        x_target[mel_mask] = PAD_TOKEN
        y = self(x)
        return F.cross_entropy(y.view(-1, N_TOKENS), x_target.view(-1), ignore_index=PAD_TOKEN)


In [ ]:
batch_data_path = 'abnormal_event_records/batch_data_global_step_23782_20250708_115616.pt'

batch_data = torch.load(batch_data_path, map_location=torch.device('cpu'))

In [55]:
batch_data['mel_data'][0]

tensor([[  0,  83,   0,  ..., 255, 255, 255],
        [  0,  83,   0,  ..., 255, 255, 255],
        [  0,  84,   0,  ..., 255, 255, 255],
        ...,
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255]], dtype=torch.uint8)

In [58]:
np.mean(batch_data['mel_data'].numpy() == batch_data['acc_data'].numpy())

np.float64(1.0)

In [56]:
np.unique(batch_data['mel_data'][0].numpy(), return_counts=True)

(array([  0,   1,   2,   3,   4,   5,   6,  48,  52,  53,  55,  57,  58,
         59,  60,  62,  64,  65,  67,  69,  71,  72,  74,  75,  76,  77,
         79,  83,  84,  86,  89,  91, 254, 255], dtype=uint8),
 array([  483,    24,    17,    20,     4,     2,     6,     3,     5,
           10,    18,     9,     6,     1,    43,    24,    19,    19,
           22,     5,    14,    22,    19,     1,     9,     5,     7,
            9,     4,     2,     1,     1,   384, 10302]))

In [ ]:
# for id, samp in enumerate(batch_data['mel_data']):
#     decode_output(
#         [samp[i, :] for i in range(samp.shape[0])],
#         f'temp/{id}_mel.mid',
#         single=True,
#         tempo=90.0,
#     )
# for id, samp in enumerate(batch_data['acc_data']):
#     decode_output(
#         [samp[i, :] for i in range(samp.shape[0])],
#         f'temp/{id}_acc.mid',
#         single=True,
#         tempo=90.0,
#     )

In [ ]:
# Create a proper decoding function for RAW data format
def decode_raw_output(raw_data, save_path, tempo=120.0):
    """
    Decode raw MIDI data (before model preprocessing) to MIDI file.
    
    Args:
        raw_data: torch.Tensor of shape [time_steps, polyphony*3] 
                 Format: [program, pitch, duration] triplets
        save_path: Path to save the MIDI file
        tempo: Tempo for the MIDI file
    """
    import pretty_midi
    from preprocess.preprocess_midi2pt_dataset import DURATION_TEMPLATES
    
    midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    time_step_length = 60.0 / tempo / 4
    
    # Reshape data: [time_steps, polyphony, 3] where 3 = [program, pitch, duration]
    time_steps, total_features = raw_data.shape
    polyphony = total_features // 3
    data = raw_data.view(time_steps, polyphony, 3)
    
    instrument_map = {}
    
    for time_step in range(time_steps):
        start_time = time_step * time_step_length
        
        for note_idx in range(polyphony):
            program = int(data[time_step, note_idx, 0].item())
            pitch = int(data[time_step, note_idx, 1].item())
            duration_idx = int(data[time_step, note_idx, 2].item())
            
            # Skip padding (255) and EOS (254) tokens
            if pitch == 255 or program == 254:
                continue
                
            # Validate values
            if pitch < 0 or pitch >= 128:
                continue
            if duration_idx < 0 or duration_idx >= len(DURATION_TEMPLATES):
                continue
                
            # Calculate end time
            duration = DURATION_TEMPLATES[duration_idx]
            end_time = start_time + duration * time_step_length
            
            # Create instrument if needed
            if program not in instrument_map:
                if program == 127:  # Drums
                    inst = pretty_midi.Instrument(program=0, is_drum=True, name="Drums")
                elif program == 0:
                    inst = pretty_midi.Instrument(program=24, name="Guitar")
                elif program == 1:
                    inst = pretty_midi.Instrument(program=0, name="Piano")
                else:
                    inst = pretty_midi.Instrument(program=program, name=f"Instrument_{program}")
                instrument_map[program] = inst
                midi.instruments.append(inst)
            
            # Add note
            inst = instrument_map[program]
            note = pretty_midi.Note(velocity=100, pitch=pitch, start=start_time, end=end_time)
            inst.notes.append(note)
    
    # Create directory and save
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    midi.write(save_path)
    print(f"Saved: {save_path}")

# Now decode the raw data correctly
print("=== Converting Raw Data to MIDI ===")
for id, samp in enumerate(batch_data['mel_data']):
    decode_raw_output(samp, f'temp/{id}_mel_raw.mid', tempo=90.0)
    
for id, samp in enumerate(batch_data['acc_data']):
    decode_raw_output(samp, f'temp/{id}_acc_raw.mid', tempo=90.0)

print("\n=== COMPARISON: What your original code was doing ===")
print("❌ Your original call: decode_output([samp[i, :] for i in range(samp.shape[0])], ...)")
print("   - Creates 384 timesteps, each with 30 values")
print("   - But decode_output expects [program, pitch_duration] pairs")
print("   - Your data has [program, pitch, duration] triplets")
print("   - This causes misalignment and incorrect MIDI generation")

print("\n✅ Solution: decode_raw_output() properly handles the raw triplet format")

# Verify the files were created
import glob
created_files = glob.glob('temp/*_raw.mid')
print(f"\n✅ Successfully created {len(created_files)} corrected MIDI files:")
for f in sorted(created_files):
    print(f"  - {f}")

In [14]:
og_data_mel = '../data/aria_deduped_skyline_top2/deduped_mel_cp4.pt'
og_data_acc = '../data/aria_deduped_skyline_top2/deduped_acc_cp4.pt'

og_mel = torch.load(og_data_mel, map_location=torch.device('cpu'))[:384]#.numpy()
og_acc = torch.load(og_data_acc, map_location=torch.device('cpu'))[:384]#.numpy()
og_mel.shape, og_acc.shape

(torch.Size([660549775, 12]), torch.Size([660446822, 12]))

In [48]:
hparam_path = 'old_m2a_transformer_aria_skyline_v0_0.5B-1.0.yaml'
with open(hparam_path, 'r') as f:
    hparams = yaml.safe_load(f)['model']

model = OldM2ATransformer(**hparams)
model_path = 'ckpt/epoch=00-val_loss=0.98.ckpt'
checkpoint = torch.load(model_path, map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['state_dict'])
model.eval()

# model = RoFormerSymbolicTransformer.load_from_checkpoint(model_path, large=False)
x_mel, x_acc = decompress(model, og_mel, og_acc)

(torch.Size([384, 12]), torch.Size([384, 12]))

In [49]:
# for id, samp in enumerate(batch_data['mel_data']):
#     decode_raw_output(samp, f'temp/{id}_mel_raw.mid', tempo=90.0)
    
# for id, samp in enumerate(batch_data['acc_data']):
#     decode_raw_output(samp, f'temp/{id}_acc_raw.mid', tempo=90.0)

/home/ubuntu/ugrip/andrew/StreamMUSE/abnormal_model/../m2a_transformer_inference.py:58: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(byte_arr_mel).unsqueeze(0)
/home/ubuntu/ugrip/andrew/StreamMUSE/abnormal_model/../m2a_transformer_inference.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(byte_arr_acc).unsqueeze(0)


In [43]:
og_mel

tensor([[254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        ...,
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255]], dtype=torch.uint8)

In [46]:
mel = og_mel.numpy()
acc = og_acc.numpy()

mel_check = mel < 254
acc_check = acc < 254
np.mean(mel_check == acc_check)

np.float64(0.892578125)